# scRNA-seq Analysis Template

| Field | Value |
|---|---|
| **Sample** | PBMC 3k (10x Genomics) |
| **Date** | *(fill in)* |
| **Author** | CH |
| **Kernel** | env_core |
| **Dataset** | scanpy built-in PBMC 3k |
| **Notes** | Template validation run |

---

## Workflow overview

1. Imports and configuration
2. Data loading
3. Quality control (QC)
4. Filtering
5. Doublet detection
6. Normalisation and feature selection
7. Dimensionality reduction
8. Clustering
9. Marker identification
10. Automated annotation
11. Summary export

All decisions are recorded in `summaries/scrna_summary.json` for the combined report.

---
## 1. Imports and configuration

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import median_abs_deviation
from datetime import datetime
import session_info2

warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────────────────
# All stochastic steps (UMAP, Leiden, scVI) use this seed.
# Record it — results are only reproducible with the same seed.
SEED = 42
np.random.seed(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Centralise all output paths here so nothing is hardcoded downstream.
# ── CHANGE THIS FOR EVERY RUN ─────────────────────────────────────────────────
SAMPLE_ID = "sampleID"   # e.g. "sample01_lung", "pbmc3k_cpu", "pbmc3k_gpu"
BASE_DIR    = f"/home/jovyan/work/analyses/{SAMPLE_ID}"
FIG_DIR     = f"{BASE_DIR}/figures"
CKPT_DIR    = f"{BASE_DIR}/checkpoints"
SUMMARY_DIR = f"{BASE_DIR}/summaries"

for d in [FIG_DIR, CKPT_DIR, SUMMARY_DIR]:
    os.makedirs(d, exist_ok=True)

sc.settings.figdir = FIG_DIR
sc.settings.verbosity = 2        # warnings + basic info

# ── Analysis parameters ───────────────────────────────────────────────────────
# Keep all tunable parameters here. When you change a threshold, change it
# here, not buried in the analysis cells.
PARAMS = {
    "sample_id"         : SAMPLE_ID,
    "seed"              : SEED,
    "date"              : datetime.today().strftime('%Y-%m-%d'),
    "author"            : "CH",
    # QC — set to None to use MAD-based automatic thresholds
    "min_genes"         : None,    # minimum genes per cell
    "max_genes"         : None,    # maximum genes per cell (doublet proxy)
    "min_counts"        : None,    # minimum UMI counts
    "max_counts"        : None,    # maximum UMI counts
    "max_pct_mt"        : None,    # maximum % mitochondrial
    "max_pct_hb"        : None,    # maximum % haemoglobin
    "max_pct_ribo"      : None,    # maximum % ribosomal (optional)
    "mad_nmads"         : 5,       # number of MADs for automatic thresholds
    # Feature selection
    "n_hvgs"            : 2000,
    "hvg_flavor"        : "seurat_v3",
    # Dimensionality reduction
    "n_pcs"             : 50,
    "n_pcs_use"         : 30,      # PCs to use for neighbours (set after elbow)
    "n_neighbours"      : 30,
    # Clustering
    "leiden_resolutions": [0.3, 0.5, 0.8, 1.0, 1.5],
    "leiden_final"      : 0.5,     # chosen resolution (set after clustree)
    # Doublet detection
    "run_solo"          : True,    # SOLO doublet detection via scvi-tools
    "solo_max_epochs"   : 400,     # training epochs (reduce for speed)
    "solo_threshold"    : 0.5,     # doublet score threshold (0-1)
    # Annotation
    "run_celltypist"    : True,
    "celltypist_model"  : "Immune_All_Low.pkl",
}

print("Configuration loaded.")
print(f"  Sample:   {PARAMS['sample_id']}")
print(f"  Date:     {PARAMS['date']}")
print(f"  Seed:     {PARAMS['seed']}")
print(f"  Base dir: {BASE_DIR}")

---
## 2. Data loading

We start from the count matrix — raw UMI counts, cells × genes. 
All QC and filtering is performed on raw counts before any transformation.

In [ ]:
# ── Load data ────────────────────────────────────────────────────────────────
# Replace sc.datasets.pbmc3k() with sc.read_10x_h5() or sc.read_10x_mtx()
# for real data. Always start from raw/unfiltered output when possible.
adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()

# Store original count matrix in .raw before any modification.
# Required later for DE, which must run on untransformed counts.
adata.layers["counts"] = adata.X.copy()

# ── Cell count tracking ───────────────────────────────────────────────────────
# This table is updated at every filtering step.
# It provides a full audit trail of how many cells were removed and why.
cell_counts = pd.DataFrame(columns=["step", "n_cells", "n_genes", "cells_removed", "notes"])

def record_counts(adata, step, notes="", prev_n=None):
    """Append current cell/gene counts to the tracking table."""
    removed = (prev_n - adata.n_obs) if prev_n is not None else 0
    row = pd.DataFrame([{
        "step"          : step,
        "n_cells"       : adata.n_obs,
        "n_genes"       : adata.n_vars,
        "cells_removed" : removed,
        "notes"         : notes,
    }])
    return pd.concat([cell_counts, row], ignore_index=True)

cell_counts = record_counts(adata, "Raw", "Input count matrix")

print(f"Loaded: {adata.n_obs} cells × {adata.n_vars} genes")
print(adata)

---
## 3. Quality control

### 3.1 Flag QC gene sets

Three gene sets flag technical artefacts:
- **Mitochondrial (mt-)**: high % indicates dying or damaged cells with broken membranes.
- **Haemoglobin (HBA/HBB)**: high % indicates red blood cell contamination.
- **Ribosomal (RPS/RPL)**: high % can indicate stressed cells; included for completeness but not filtered by default.

Gene name prefixes vary by species — check before running on non-human/non-mouse data.

In [ ]:
# ── Flag QC gene sets ─────────────────────────────────────────────────────────
# Human: MT- / HBA / HBB / RPS / RPL
# Mouse: mt- / Hba / Hbb / Rps / Rpl
# Adjust prefixes for your organism.

adata.var["mt"]   = adata.var_names.str.startswith("MT-")
adata.var["hb"]   = adata.var_names.str.startswith(("HBA", "HBB"))
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))

print(f"Mitochondrial genes : {adata.var['mt'].sum()}")
print(f"Haemoglobin genes   : {adata.var['hb'].sum()}")
print(f"Ribosomal genes     : {adata.var['ribo'].sum()}")

### 3.2 Calculate QC metrics

Metrics are calculated on raw counts (`adata.X` at this point). 
Key metrics per cell: `total_counts`, `n_genes_by_counts`, `pct_counts_mt`, `pct_counts_hb`, `pct_counts_ribo`.

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "hb", "ribo"],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("── QC summary ──────────────────────────────────────────")
print(adata.obs[[
    "total_counts",
    "n_genes_by_counts",
    "pct_counts_mt",
    "pct_counts_hb",
    "pct_counts_ribo"
]].describe().round(2))

### 3.3 MAD-based threshold calculation

Thresholds are computed as `median ± N × MAD` (N = `mad_nmads` in PARAMS).
This is sample-adaptive — a threshold that works for one sample will not necessarily 
work for another. If `PARAMS` contains a non-None value, the manual override is used instead.

Per Heumos et al. 2023: thresholds should be set per-sample, then reassessed after 
downstream clustering if low-quality cells cluster together.

In [ ]:
def mad_threshold(series, nmads=5, lower=True, upper=True):
    """Compute MAD-based lower and/or upper threshold for a metric series."""
    med = np.median(series)
    mad = median_abs_deviation(series)
    lo  = med - nmads * mad if lower else -np.inf
    hi  = med + nmads * mad if upper else  np.inf
    return lo, hi, med, mad

nmads = PARAMS["mad_nmads"]

# Compute MAD thresholds for each metric
lo_genes,  hi_genes,  _, _ = mad_threshold(adata.obs["n_genes_by_counts"],  nmads)
lo_counts, hi_counts, _, _ = mad_threshold(adata.obs["total_counts"],       nmads)
_,         hi_mt,     _, _ = mad_threshold(adata.obs["pct_counts_mt"],      nmads, lower=False)
_,         hi_hb,     _, _ = mad_threshold(adata.obs["pct_counts_hb"],      nmads, lower=False)

# Apply manual overrides from PARAMS if set
thresholds = {
    "min_genes"  : PARAMS["min_genes"]   if PARAMS["min_genes"]   else max(200, lo_genes),
    "max_genes"  : PARAMS["max_genes"]   if PARAMS["max_genes"]   else hi_genes,
    "min_counts" : PARAMS["min_counts"]  if PARAMS["min_counts"]  else max(500, lo_counts),
    "max_counts" : PARAMS["max_counts"]  if PARAMS["max_counts"]  else hi_counts,
    "max_pct_mt" : PARAMS["max_pct_mt"]  if PARAMS["max_pct_mt"]  else min(20, hi_mt),
    "max_pct_hb" : PARAMS["max_pct_hb"]  if PARAMS["max_pct_hb"]  else min(5,  hi_hb),
}

print(f"QC thresholds (MAD method, N={nmads}):")
for k, v in thresholds.items():
    src = "manual" if PARAMS.get(k) else "MAD-derived"
    print(f"  {k:<14}: {v:.1f}  ({src})")

# Count cells that would be removed at each threshold
n_low_genes  = (adata.obs["n_genes_by_counts"]  < thresholds["min_genes"]).sum()
n_high_genes = (adata.obs["n_genes_by_counts"]  > thresholds["max_genes"]).sum()
n_low_counts = (adata.obs["total_counts"]        < thresholds["min_counts"]).sum()
n_high_counts= (adata.obs["total_counts"]        > thresholds["max_counts"]).sum()
n_high_mt    = (adata.obs["pct_counts_mt"]       > thresholds["max_pct_mt"]).sum()
n_high_hb    = (adata.obs["pct_counts_hb"]       > thresholds["max_pct_hb"]).sum()

print(f"\nCells failing each threshold (may overlap):")
print(f"  n_genes < {thresholds['min_genes']:.0f}      : {n_low_genes}")
print(f"  n_genes > {thresholds['max_genes']:.0f}    : {n_high_genes}")
print(f"  counts  < {thresholds['min_counts']:.0f}     : {n_low_counts}")
print(f"  counts  > {thresholds['max_counts']:.0f}   : {n_high_counts}")
print(f"  pct_mt  > {thresholds['max_pct_mt']:.1f}%     : {n_high_mt}")
print(f"  pct_hb  > {thresholds['max_pct_hb']:.1f}%      : {n_high_hb}")

### 3.4 QC visualisation — distributions

Inspect distributions before committing to thresholds. 
Red dashed lines = computed thresholds. Adjust `PARAMS` if thresholds look wrong, then re-run from 3.3.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle(f"QC distributions — {SAMPLE_ID}", fontsize=14, y=1.01)

metrics = [
    ("total_counts",        "Total UMI counts",          None,  thresholds["min_counts"], thresholds["max_counts"]),
    ("n_genes_by_counts",   "Genes detected",            None,  thresholds["min_genes"],  thresholds["max_genes"]),
    ("pct_counts_mt",       "% mitochondrial",           None,  None,                     thresholds["max_pct_mt"]),
    ("pct_counts_hb",       "% haemoglobin",             None,  None,                     thresholds["max_pct_hb"]),
    ("pct_counts_ribo",     "% ribosomal",               None,  None,                     None),
    ("total_counts",        "Total counts (log scale)",  "log", thresholds["min_counts"], thresholds["max_counts"]),
]

scatter_pairs = [
    ("total_counts", "n_genes_by_counts",  "UMI vs genes",          thresholds["min_counts"], thresholds["min_genes"]),
    ("total_counts", "pct_counts_mt",      "UMI vs % mt",           thresholds["min_counts"], thresholds["max_pct_mt"]),
    ("n_genes_by_counts", "pct_counts_mt", "Genes vs % mt",         thresholds["min_genes"],  thresholds["max_pct_mt"]),
]

for ax, (metric, title, yscale, lo, hi) in zip(axes.flat[:6], metrics):
    ax.hist(adata.obs[metric], bins=80, color="steelblue", alpha=0.75, edgecolor="none")
    if lo: ax.axvline(lo, color="red",    ls="--", lw=1.5, label=f"min={lo:.0f}")
    if hi: ax.axvline(hi, color="orange", ls="--", lw=1.5, label=f"max={hi:.0f}")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(metric)
    ax.set_ylabel("Cells")
    if yscale == "log": ax.set_yscale("log")
    if lo or hi: ax.legend(fontsize=8)

for ax, (xm, ym, title, xlo, ylo) in zip(axes.flat[6:], scatter_pairs):
    ax.scatter(adata.obs[xm], adata.obs[ym], s=1, alpha=0.3, c="steelblue", rasterized=True)
    if xlo: ax.axvline(xlo, color="red",    ls="--", lw=1.2)
    if ylo: ax.axhline(ylo, color="orange", ls="--", lw=1.2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xm); ax.set_ylabel(ym)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/01_qc_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 01_qc_distributions.png")

### 3.5 QC visualisation — violin plots

Violin plots show the full distribution shape. 
For multi-sample datasets, set `groupby` to your sample/batch key.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 5))
fig.suptitle(f"QC metrics — {SAMPLE_ID}", fontsize=13)

qc_metrics = [
    ("total_counts",      "Total UMI",     thresholds["min_counts"], thresholds["max_counts"]),
    ("n_genes_by_counts", "Genes/cell",    thresholds["min_genes"],  thresholds["max_genes"]),
    ("pct_counts_mt",     "% mt",          None,                     thresholds["max_pct_mt"]),
    ("pct_counts_hb",     "% hb",          None,                     thresholds["max_pct_hb"]),
    ("pct_counts_ribo",   "% ribo",        None,                     None),
]

for ax, (metric, label, lo, hi) in zip(axes, qc_metrics):
    vals = adata.obs[metric]
    parts = ax.violinplot(vals, showmedians=True)
    parts["bodies"][0].set_facecolor("steelblue")
    parts["bodies"][0].set_alpha(0.7)
    if lo: ax.axhline(lo, color="red",    ls="--", lw=1.5, label=f"min={lo:.0f}")
    if hi: ax.axhline(hi, color="orange", ls="--", lw=1.5, label=f"max={hi:.1f}")
    ax.set_title(label)
    ax.set_xticks([])
    if lo or hi: ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/02_qc_violins.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 02_qc_violins.png")

---
## 4. Filtering

Filtering order:
1. Remove genes expressed in <3 cells (reduces memory; does not affect QC metrics already calculated).
2. Apply cell-level QC thresholds one at a time, recording counts after each step.

Per Luecken & Theis 2019: start permissive and tighten after downstream clustering 
if low-quality cells cluster separately.

In [ ]:
# ── Gene filtering ────────────────────────────────────────────────────────────
# Remove genes detected in fewer than 3 cells.
# This does not remove biologically rare genes — it removes noise.
n_before = adata.n_obs
sc.pp.filter_genes(adata, min_cells=3)
print(f"Gene filter (min_cells=3): {adata.n_vars} genes retained")

# ── Cell filtering ────────────────────────────────────────────────────────────
filters = [
    ("n_genes_by_counts",  ">=", thresholds["min_genes"],   "low genes"),
    ("n_genes_by_counts",  "<=", thresholds["max_genes"],   "high genes (putative doublets)"),
    ("total_counts",       ">=", thresholds["min_counts"],  "low counts"),
    ("total_counts",       "<=", thresholds["max_counts"],  "high counts (putative doublets)"),
    ("pct_counts_mt",      "<=", thresholds["max_pct_mt"],  "high mt% (dying cells)"),
    ("pct_counts_hb",      "<=", thresholds["max_pct_hb"],  "high hb% (RBC contamination)"),
]

cell_counts = record_counts(adata, "Post gene filter", "min_cells=3", n_before)

for metric, op, threshold, reason in filters:
    n_before = adata.n_obs
    if op == ">=":
        adata = adata[adata.obs[metric] >= threshold].copy()
    elif op == "<=":
        adata = adata[adata.obs[metric] <= threshold].copy()
    cell_counts = record_counts(
        adata,
        f"Filter: {reason}",
        f"{metric} {op} {threshold:.1f}",
        n_before
    )
    print(f"  {reason:<38}: {adata.n_obs:>5} cells remaining  (removed {n_before - adata.n_obs})")

print(f"\nPost-filtering: {adata.n_obs} cells × {adata.n_vars} genes")
print("\n── Cell count tracking ──────────────────────────────────")
print(cell_counts.to_string(index=False))

### 4.1 Post-filter QC check

Visually confirm filtering removed the expected tails.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Post-filter QC", fontsize=12)

for ax, (xm, ym, title) in zip(axes, [
    ("total_counts",      "n_genes_by_counts", "UMI vs genes"),
    ("total_counts",      "pct_counts_mt",     "UMI vs % mt"),
    ("n_genes_by_counts", "pct_counts_mt",     "Genes vs % mt"),
]):
    ax.scatter(adata.obs[xm], adata.obs[ym], s=1, alpha=0.3, c="steelblue", rasterized=True)
    ax.set_title(title); ax.set_xlabel(xm); ax.set_ylabel(ym)

plt.tight_layout()
plt.savefig(f"{FIG_DIR}/03_qc_postfilter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: 03_qc_postfilter.png")

# ── Checkpoint ────────────────────────────────────────────────────────────────
adata.write(f"{CKPT_DIR}/01_filtered.h5ad")
print(f"Checkpoint saved: 01_filtered.h5ad")

---
## 5. Doublet detection

Doublets are droplets containing two cells. They appear as clusters with mixed transcriptomes 
and inflate apparent cell heterogeneity.

**SOLO** (Bernstein et al. 2020) trains a variational autoencoder on the data, simulates 
doublets in latent space, and classifies real cells by similarity to simulated doublets. 
It outperforms Scrublet on most datasets (Xi & Li 2021 benchmark) and runs natively 
on GPU via scvi-tools.

Cells are **flagged but not removed** here — confirm they form a distinct cluster in UMAP 
before removing. If they do not cluster separately, they may be rare cell types rather 
than technical doublets.

In [ ]:
if PARAMS["run_solo"]:
    import scvi

    scvi.settings.seed = SEED

    # SOLO requires an scVI model trained on the data.
    # We train a lightweight model here specifically for doublet detection.
    # This is separate from any scVI model used for batch correction downstream.
    scvi.model.SCVI.setup_anndata(adata, layer="counts")

    vae = scvi.model.SCVI(
        adata,
        n_layers=2,
        n_latent=30,
        gene_likelihood="nb"        # negative binomial — appropriate for UMI counts
    )
    vae.train(
        max_epochs=PARAMS["solo_max_epochs"],
        train_size=0.9,
        validation_size=0.1,
        early_stopping=True,
    )

    # Train SOLO on top of the scVI model
    solo = scvi.external.SOLO.from_scvi_model(vae)
    solo.train(max_epochs=200, early_stopping=True)

    # Get doublet predictions
    solo_results = solo.predict()
    adata.obs["doublet_score"]     = solo_results["doublet"].values
    adata.obs["predicted_doublet"] = adata.obs["doublet_score"] > PARAMS["solo_threshold"]

    n_doublets   = adata.obs["predicted_doublet"].sum()
    pct_doublets = 100 * n_doublets / adata.n_obs

    print(f"SOLO doublet detection:")
    print(f"  Predicted doublets: {n_doublets} ({pct_doublets:.1f}%)")
    print(f"  Threshold: {PARAMS["solo_threshold"]}")

    # Score distribution
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(adata.obs["doublet_score"], bins=60, color="steelblue", alpha=0.75, edgecolor="none")
    ax.axvline(PARAMS["solo_threshold"], color="red", ls="--", lw=1.5,
               label=f"threshold={PARAMS["solo_threshold"]}")
    ax.set_xlabel("SOLO doublet score")
    ax.set_ylabel("Cells")
    ax.set_title("SOLO doublet score distribution")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{FIG_DIR}/04_solo_doublets.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: 04_solo_doublets.png")

    cell_counts = record_counts(
        adata, "Post doublet detection",
        f"SOLO flagged {n_doublets} ({pct_doublets:.1f}%) — not yet removed"
    )
else:
    print("SOLO skipped (run_solo=False).")
    adata.obs["doublet_score"]     = np.nan
    adata.obs["predicted_doublet"] = False
    n_doublets   = 0
    pct_doublets = 0.0

---
## 6. Normalisation and feature selection

### 6.1 Normalisation

Library-size normalisation corrects for differences in sequencing depth between cells. 
Each cell is scaled to 10,000 counts (CPM-like), then log1p-transformed.

Raw counts are preserved in `adata.layers['counts']` for downstream DE analysis, 
which requires untransformed integer counts.

In [ ]:
# Normalise to 10,000 counts per cell
sc.pp.normalize_total(adata, target_sum=1e4)

# Log1p transform: log(x+1) — compresses dynamic range, prevents large counts
# from dominating. The +1 avoids log(0) for zero-count genes.
sc.pp.log1p(adata)

# Store normalised values for later reference
adata.layers["normalised"] = adata.X.copy()

print("Normalisation complete.")
print(f"  X range: {adata.X.min():.2f} – {adata.X.max():.2f}")

### 6.2 Highly variable gene selection

HVG selection retains genes that vary across cells more than expected by chance. 
This focuses downstream analysis on biologically informative genes and reduces noise.

`seurat_v3` flavour uses a negative binomial model fitted to raw counts — 
it is more robust than the original Seurat method for datasets with variable sequencing depth.

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=PARAMS["n_hvgs"],
    flavor=PARAMS["hvg_flavor"],
    layer="counts",          # use raw counts for seurat_v3
    subset=False             # flag HVGs, do not subset yet
)

n_hvg = adata.var["highly_variable"].sum()
print(f"Highly variable genes: {n_hvg} / {adata.n_vars}")

# HVG mean-variance plot
sc.pl.highly_variable_genes(
    adata,
    save=f"_{SAMPLE_ID}_hvg.png",
    show=True
)

# Checkpoint
adata.write(f"{CKPT_DIR}/02_normalised.h5ad")
print("Checkpoint saved: 02_normalised.h5ad")

---
## 7. Dimensionality reduction

### 7.1 PCA

PCA on HVGs only. Scaling (zero mean, unit variance) is applied within PCA to prevent 
highly expressed genes from dominating. `max_value=10` clips extreme outliers.

In [ ]:
# Scale before PCA — clips to max_value=10 to limit outlier influence
sc.pp.scale(adata, max_value=10)

sc.tl.pca(
    adata,
    n_comps=PARAMS["n_pcs"],
    use_highly_variable=True,
    svd_solver="arpack",
    random_state=SEED
)

# Elbow plot — look for where variance explained flattens.
# Use n_pcs_use PCs for neighbour graph. Adjust PARAMS['n_pcs_use'] accordingly.
sc.pl.pca_variance_ratio(
    adata,
    n_pcs=PARAMS["n_pcs"],
    log=True,
    save=f"_{SAMPLE_ID}_elbow.png",
    show=True
)

print(f"PCA complete. Using top {PARAMS['n_pcs_use']} PCs for downstream analysis.")
print("Adjust PARAMS['n_pcs_use'] based on the elbow plot before continuing.")

### 7.2 Neighbour graph and UMAP

The KNN graph is the basis for both clustering and UMAP. 
`n_pcs_use` controls how many PCs are used — set from elbow plot above.

UMAP is for visualisation only. Do not draw biological conclusions from 2D embedding 
distance alone (Luecken & Theis 2019 — quantitative assessments should complement visual inspection).

In [ ]:
sc.pp.neighbors(
    adata,
    n_pcs=PARAMS["n_pcs_use"],
    n_neighbors=PARAMS["n_neighbours"],
    random_state=SEED
)

sc.tl.umap(adata, random_state=SEED)

# Basic UMAP — coloured by QC metrics to spot residual artefacts
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "pct_counts_mt", "predicted_doublet", "doublet_score"],
    ncols=3,
    save=f"_{SAMPLE_ID}_umap_qc.png",
    show=True
)

# Checkpoint
adata.write(f"{CKPT_DIR}/03_reduced.h5ad")
print("Checkpoint saved: 03_reduced.h5ad")

---
## 8. Clustering

### 8.1 Leiden at multiple resolutions

Leiden is run at multiple resolutions. Resolution controls cluster granularity — 
higher = more clusters. The final resolution is chosen after examining:
1. Clustree stability plot below
2. Whether clusters correspond to known biology

Leiden is preferred over Louvain because it guarantees well-connected communities 
(Traag et al. 2019). Per Heumos et al. 2023: do not commit to one resolution — 
subclustering specific populations later is valid.

In [ ]:
resolution_results = {}

for res in PARAMS["leiden_resolutions"]:
    key = f"leiden_{res}"
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        random_state=SEED
    )
    n_clusters = adata.obs[key].nunique()
    resolution_results[res] = n_clusters
    print(f"  res={res:<5}: {n_clusters:>3} clusters")

# Plot all resolutions on UMAP
sc.pl.umap(
    adata,
    color=[f"leiden_{r}" for r in PARAMS["leiden_resolutions"]],
    ncols=3,
    legend_loc="on data",
    save=f"_{SAMPLE_ID}_umap_leiden_resolutions.png",
    show=True
)

### 8.2 Clustree — resolution stability

Clustree traces how cells move between clusters as resolution increases. 
Stable transitions (straight lines) indicate robust clusters. 
Large jumps or reversed flows indicate instability at that resolution.

In [ ]:
from pyclustree import clustree as pyclustree_plot

cluster_keys = [f"leiden_{r}" for r in PARAMS["leiden_resolutions"]]

try:
    fig = pyclustree_plot(
        adata,
        cluster_keys=cluster_keys,
        title=f"Clustree — {SAMPLE_ID}",
    )
    fig.savefig(f"{FIG_DIR}/08_clustree.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: 08_clustree.png")
except Exception as e:
    print(f"pyclustree failed: {e}")
    print("Resolution summary:")
    for res, n in resolution_results.items():
        print(f"  res={res}: {n} clusters")

In [ ]:
# ── Set final clustering resolution ──────────────────────────────────────────
# After inspecting clustree and the resolution UMAPs above,
# set PARAMS['leiden_final'] and run this cell.

final_key = f"leiden_{PARAMS['leiden_final']}"
adata.obs["leiden"] = adata.obs[final_key].copy()

n_final_clusters = adata.obs["leiden"].nunique()
print(f"Final clustering: resolution={PARAMS['leiden_final']}, {n_final_clusters} clusters")

sc.pl.umap(
    adata,
    color="leiden",
    legend_loc="on data",
    title=f"Leiden r={PARAMS['leiden_final']} ({n_final_clusters} clusters)",
    save=f"_{SAMPLE_ID}_umap_final_leiden.png",
    show=True
)

# Checkpoint
adata.write(f"{CKPT_DIR}/04_clustered.h5ad")
print("Checkpoint saved: 04_clustered.h5ad")

---
## 9. Marker identification

### 9.1 Rank genes by cluster

Wilcoxon rank-sum test: each cluster vs all others. 
Genes ranked by score (combines fold-change and significance). 

**Important**: DE is run on `raw` counts (layer='counts'), not on scaled/normalised values. 
Scaled values compress variance and are unsuitable for statistical testing.

Per Luecken & Theis 2019: p-values from clustering-derived groups are inflated 
because labels and data are not independent. Use markers for annotation, not publication-grade DE.

In [ ]:
# Restore normalised (not scaled) counts for DE
adata.X = adata.layers["normalised"].copy()

sc.tl.rank_genes_groups(
    adata,
    groupby="leiden",
    method="wilcoxon",
    key_added="markers",
    use_raw=False,
    corr_method="benjamini-hochberg"
)

# Top 5 markers per cluster
sc.pl.rank_genes_groups(
    adata,
    n_genes=5,
    key="markers",
    sharey=False,
    save=f"_{SAMPLE_ID}_markers_dotplot.png",
    show=True
)

# Extract to DataFrame for summary export
markers_df = sc.get.rank_genes_groups_df(
    adata, group=None, key="markers", pval_cutoff=0.05, log2fc_min=0.5
)
print(f"\nSignificant markers (p<0.05, logFC>0.5): {len(markers_df)}")
print(markers_df.groupby("group").head(3).to_string())

### 9.2 Dot plot — canonical markers

Overlay canonical PBMC markers to anchor cluster identity. 
Adjust `canonical_markers` for your cell types / organism.

In [ ]:
# Canonical PBMC markers (human)
canonical_markers = {
    "CD4 T"    : ["CD3D", "CD4", "IL7R"],
    "CD8 T"    : ["CD3D", "CD8A", "CD8B"],
    "NK"       : ["GNLY", "NKG7", "NCAM1"],
    "B cell"   : ["MS4A1", "CD79A", "CD19"],
    "Monocyte" : ["CD14", "LYZ", "S100A9"],
    "DC"       : ["FCER1A", "CST3"],
    "Platelet" : ["PPBP", "PF4"],
}

marker_genes_flat = [g for genes in canonical_markers.values() for g in genes]
# Keep only genes present in the dataset
marker_genes_flat = [g for g in marker_genes_flat if g in adata.var_names]

if marker_genes_flat:
    sc.pl.dotplot(
        adata,
        marker_genes_flat,
        groupby="leiden",
        standard_scale="var",
        title="Canonical PBMC markers",
        save=f"_{SAMPLE_ID}_canonical_markers.png",
        show=True
    )
else:
    print("No canonical markers found in dataset.")

---
## 10. Automated cell type annotation — CellTypist

CellTypist uses a pre-trained logistic regression model to annotate cell types. 
It is fast and well-suited for immune cell populations.

Two modes:
- `majority_voting=False`: per-cell predictions
- `majority_voting=True`: smooths predictions within Leiden clusters — 
  recommended when clusters are biologically meaningful

Results should be verified against canonical marker expression (section 9.2). 
Automated annotation is a starting point, not ground truth.

In [ ]:
if PARAMS["run_celltypist"]:
    import celltypist
    from celltypist import models

    # Download model if not cached
    model = models.Model.load(model=PARAMS["celltypist_model"])

    # CellTypist expects normalised log1p counts (not scaled)
    adata_ct = adata.copy()
    adata_ct.X = adata_ct.layers["normalised"].copy()

    predictions = celltypist.annotate(
        adata_ct,
        model=model,
        majority_voting=True,
        over_clustering="leiden"
    )

    # Transfer annotations back to main adata
    adata.obs["celltypist_cell_type"]          = predictions.predicted_labels["majority_voting"]
    adata.obs["celltypist_conf_score"]         = predictions.probability_matrix.max(axis=1).values

    print("CellTypist annotation summary:")
    print(adata.obs["celltypist_cell_type"].value_counts().to_string())

    sc.pl.umap(
        adata,
        color=["celltypist_cell_type", "celltypist_conf_score"],
        ncols=2,
        save=f"_{SAMPLE_ID}_celltypist.png",
        show=True
    )
else:
    print("CellTypist skipped (run_celltypist=False).")
    adata.obs["celltypist_cell_type"] = "unknown"
    adata.obs["celltypist_conf_score"] = np.nan

---
## 11. Summary export

All key metrics are collected into a JSON file that the `report.ipynb` notebook reads. 
This is the bridge between kernels — each analysis notebook writes its own summary, 
and the report notebook assembles them without needing to re-run the analysis.

In [ ]:
# ── Final checkpoint ──────────────────────────────────────────────────────────
adata.write(f"{CKPT_DIR}/05_annotated.h5ad")
print("Checkpoint saved: 05_annotated.h5ad")

# ── Build summary dict ────────────────────────────────────────────────────────
summary = {
    "metadata": {
        "sample_id" : PARAMS["sample_id"],
        "date"      : PARAMS["date"],
        "author"    : PARAMS["author"],
        "kernel"    : "env_core",
        "notebook"  : "01_scrna_analysis.ipynb",
    },
    "params": PARAMS,
    "qc": {
        "thresholds"         : {k: round(float(v), 2) for k, v in thresholds.items()},
        "cell_counts_table"  : cell_counts.to_dict(orient="records"),
        "n_cells_raw"        : int(cell_counts.iloc[0]["n_cells"]),
        "n_cells_final"      : int(adata.n_obs),
        "n_genes_final"      : int(adata.n_vars),
        "n_hvgs"             : int(n_hvg),
    },
    "doublets": {
        "method"             : "scrublet" if PARAMS["run_solo"] else "none",
        "n_predicted"        : int(n_doublets),
        "pct_predicted"      : round(float(pct_doublets), 2),
    },
    "clustering": {
        "method"             : "leiden",
        "resolutions_tested" : PARAMS["leiden_resolutions"],
        "resolution_final"   : PARAMS["leiden_final"],
        "n_clusters"         : int(n_final_clusters),
        "n_pcs_used"         : PARAMS["n_pcs_use"],
        "n_neighbours"       : PARAMS["n_neighbours"],
    },
    "annotation": {
        "method"             : "celltypist" if PARAMS["run_celltypist"] else "none",
        "model"              : PARAMS["celltypist_model"],
        "cell_type_counts"   : adata.obs["celltypist_cell_type"].value_counts().to_dict()
                               if PARAMS["run_celltypist"] else {},
        "median_conf_score"  : round(float(adata.obs["celltypist_conf_score"].median()), 3)
                               if PARAMS["run_celltypist"] else None,
    },
    "figures": [
        "01_qc_distributions.png",
        "02_qc_violins.png",
        "03_qc_postfilter.png",
        "04_scrublet_histogram.png",
        "08_clustree.png",
    ],
}

summary_path = f"{SUMMARY_DIR}/scrna_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Summary saved: {summary_path}")
print(f"\n── Analysis complete ──────────────────────────────────────")
print(f"  Cells:    {summary['qc']['n_cells_raw']} → {summary['qc']['n_cells_final']}")
print(f"  Genes:    {summary['qc']['n_genes_final']}")
print(f"  HVGs:     {summary['qc']['n_hvgs']}")
print(f"  Clusters: {summary['clustering']['n_clusters']}")
if PARAMS["run_celltypist"]:
    print(f"  Cell types annotated: {len(summary['annotation']['cell_type_counts'])}")

---
## 12. Software versions

Recorded for reproducibility. Any re-analysis must use the same package versions 
to reproduce results exactly (Sandve et al. 2013, Rule 4).

In [ ]:
import session_info2
session_info2.session_info()